In [1]:
# =========================================================
# TECH CHALLENGE - VERSÃO COM VADER
# =========================================================

# =============================
# 1️⃣ Instalações
# =============================
!pip install -q nltk tensorflow wordcloud

import nltk
nltk.download("stopwords")
nltk.download("vader_lexicon")

# =============================
# 2️⃣ Importações
# =============================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from wordcloud import WordCloud

from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import CountVectorizer

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =============================
# 3️⃣ Carregar CSV
# =============================
df = pd.read_csv("/content/complaints_processed.csv")

print("Shape original:", df.shape)

# Opcional: reduzir tamanho para acelerar
#df = df.sample(50000, random_state=42)

# =============================
# 4️⃣ Criar Sentimento com VADER
# =============================
sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    if pd.isna(text):
        return 0

    score = sia.polarity_scores(text)["compound"]

    if score >= 0.05:
        return 1  # Positivo
    elif score <= -0.05:
        return 0  # Negativo
    else:
        return 0  # Neutro tratado como negativo (reclamações)

print("Gerando sentimento com VADER...")
df["sentiment"] = df["narrative"].apply(vader_sentiment)

print(df["sentiment"].value_counts())

# =====================
# 5️⃣ Pré-processamento
# =====================
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]

    return " ".join(tokens)

df["clean_text"] = df["narrative"].apply(preprocess_text)

# =============================
# 6️⃣ Tokenização
# =============================
MAX_FEATURES = 15000
MAX_LEN = 150

tokenizer = Tokenizer(num_words=MAX_FEATURES)
tokenizer.fit_on_texts(df["clean_text"])

sequences = tokenizer.texts_to_sequences(df["clean_text"])
X = pad_sequences(sequences, maxlen=MAX_LEN)

y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =============================
# 7️⃣ Modelo BiLSTM
# =============================
model = Sequential([
    Embedding(MAX_FEATURES, 128),
    Bidirectional(LSTM(128)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    epochs=4,
    batch_size=128,
    validation_split=0.2
)

# =============================
# 8️⃣ Avaliação
# =============================
y_pred = (model.predict(X_test) > 0.5).astype("int32")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# =============================
# 9️⃣ Curva de Accuracy
# =============================
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.title("Accuracy")
plt.legend()
plt.show()

# =============================
# 🔟 Análise das Dores
# =============================
neg_df = df[df["sentiment"] == 0]

for product in neg_df["product"].unique():
    subset = neg_df[neg_df["product"] == product]

    vectorizer = CountVectorizer(max_features=10)
    X_counts = vectorizer.fit_transform(subset["clean_text"])

    print(f"\nProduto: {product}")
    print(vectorizer.get_feature_names_out())

# =============================
# 1️⃣1️⃣ WordCloud
# =============================
text = " ".join(neg_df["clean_text"])

wordcloud = WordCloud(width=900, height=400, background_color="white").generate(text)

plt.figure(figsize=(14,6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Principais Termos - Reclamações Negativas")
plt.show()

print("\n🚀 Pipeline com VADER concluído com sucesso!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


FileNotFoundError: [Errno 2] No such file or directory: '/content/complaints_processed.csv'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Garantir que a coluna produto tenha nome consistente
# Se no seu dataset for 'Product', ajuste aqui
df.columns = df.columns.str.lower()

# Intensidade = % de reclamações negativas por produto
intensidade = (
    df.groupby("product")["sentiment"]
      .mean()
      .sort_values(ascending=False)
)

intensidade_percent = intensidade * 100

plt.figure()
plt.bar(intensidade_percent.index, intensidade_percent.values)

plt.xticks(rotation=45)
plt.ylabel("Percentual de Reclamações Negativas (%)")
plt.title("Intensidade de Reclamações por Produto")

plt.tight_layout()
plt.show()

intensidade_percent